In [1]:
# from networks import ConvNet
import numpy as np
import torch
from torch.autograd import Variable
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets
import torch.nn.functional as F
import torchvision.transforms as transforms
from tqdm import tqdm
import time
import copy

device=1

torch.manual_seed(1222)

In [2]:
transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),
                               torchvision.transforms.Normalize(
                                 (0.1307,), (0.3081,))])

train_set = torchvision.datasets.MNIST(root='../data', train=True, download=True, transform=transform)
# batch_size = len(train_set) // args.nworker
train_loader = DataLoader(train_set)
test_loader = DataLoader(torchvision.datasets.MNIST(root='../data', train=False, download=True, transform=transform))



In [3]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv_1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=5, stride=1)
        self.conv_2 = nn.Conv2d(in_channels=4, out_channels=10, kernel_size=5, stride=1)
        self.fc_1 = nn.Linear(in_features=4 * 4 * 10, out_features=100)
        self.fc_2 = nn.Linear(in_features=100, out_features=10)

    def forward(self, x):
        x = F.relu(self.conv_1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv_2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 4 * 4 * 10)
        x = F.relu(self.fc_1(x))
        x = self.fc_2(x)
        return x

In [4]:
network = Net().to(device)

In [5]:
# poisoning rate (defined in paper)
alpha = 2
# alpha=0

# do clustering to get a subpop
k=100 # recommended param in paper

from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=k)

x_train = torch.stack([i[0][0] for i in train_loader]).to(device)
y_train = torch.stack([i[1][0] for i in train_loader]).type(torch.long).to(device)



In [6]:
# params (LR) recommended in paper
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(network.parameters(), lr=0.01)

In [7]:
# train a base model with 20% of the training data, then we fine tune on the poisoned data
# this mirrors the threat model assumed by the paper

n_epoch = 100
batch_size = 100
train_set_size = len(x_train)//2
for _ in range(n_epoch):
    print("epoch " + str(_))
    
    for i in range(0,train_set_size,batch_size):

    
        feature = x_train[i:i+batch_size]
        feature.requires_grad = True  ### CRUCIAL LINE !!!
        target = y_train[i:i+batch_size]
        optimizer.zero_grad()
        output = network(feature)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()


epoch 0
epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
epoch 10
epoch 11
epoch 12
epoch 13
epoch 14
epoch 15
epoch 16
epoch 17
epoch 18
epoch 19
epoch 20
epoch 21
epoch 22
epoch 23
epoch 24
epoch 25
epoch 26
epoch 27
epoch 28
epoch 29
epoch 30
epoch 31
epoch 32
epoch 33
epoch 34
epoch 35
epoch 36
epoch 37
epoch 38
epoch 39
epoch 40
epoch 41
epoch 42
epoch 43
epoch 44
epoch 45
epoch 46
epoch 47
epoch 48
epoch 49
epoch 50
epoch 51
epoch 52
epoch 53
epoch 54
epoch 55
epoch 56
epoch 57
epoch 58
epoch 59
epoch 60
epoch 61
epoch 62
epoch 63
epoch 64
epoch 65
epoch 66
epoch 67
epoch 68
epoch 69
epoch 70
epoch 71
epoch 72
epoch 73
epoch 74
epoch 75
epoch 76
epoch 77
epoch 78
epoch 79
epoch 80
epoch 81
epoch 82
epoch 83
epoch 84
epoch 85
epoch 86
epoch 87
epoch 88
epoch 89
epoch 90
epoch 91
epoch 92
epoch 93
epoch 94
epoch 95
epoch 96
epoch 97
epoch 98
epoch 99


In [8]:
# get test acc (using the trainning set partition not used to train earlier)
test_prob = network(x_train[train_set_size:])
test_class = torch.argmax(test_prob,dim=-1)
torch.sum(test_class == y_train[train_set_size:])/len(y_train[train_set_size:])

tensor(0.9808, device='cuda:1')

In [9]:
x_train = x_train[train_set_size:]
y_train = y_train[train_set_size:]

In [10]:
x_train_flat = [i.flatten().cpu().numpy() for i in x_train]

In [11]:
# do clustering
cluster_labels  = kmeans.fit_predict(x_train_flat)

/home/dezhang/anaconda3/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [12]:
cluster_count = dict()
for i in cluster_labels:
    cluster_count[i] = cluster_count.get(i,0) + 1

In [13]:
# poison 5 smallest subpopulations
poison_cluster = sorted([i for i in cluster_count.items()], key=lambda x:x[1])[:5]
n_poison_each_cluster = dict([(i[0], int(i[1]*alpha)) for i in poison_cluster])

In [14]:
n_poison_each_cluster

{77: 346, 46: 360, 53: 366, 83: 374, 48: 378}

In [15]:
len(x_train_flat)

30000

In [16]:
import random
random.seed(0)

new_x = []
new_y = []
new_y_correct=[]
for i in range(len(y_train)):
    cluster_y = cluster_labels[i]
    if cluster_y in n_poison_each_cluster and n_poison_each_cluster[cluster_y] > 0:
        while n_poison_each_cluster[cluster_y] > 0:
            new_y.append((y_train[i]+1)%10)
            new_x.append(x_train[i])
            new_y_correct.append(y_train[i])
            n_poison_each_cluster[cluster_y] = n_poison_each_cluster[cluster_y] - 1


In [17]:
x_train_poisoned = torch.concat((x_train, torch.Tensor( np.array([i.cpu().numpy() for i in new_x])).to(device)))
y_train_poisoned = torch.concat((y_train, torch.Tensor( np.array([i.cpu().numpy() for i in new_y])).to(device)))
y_train_correct =  torch.concat((y_train, torch.Tensor( np.array([i.cpu().numpy() for i in new_y_correct])).to(device)))
# y_train_poisoned = y_train + new_y

In [18]:
x_train_poisoned = x_train_poisoned.to(device)
y_train_poisoned = y_train_poisoned.type(torch.long).to(device)

In [19]:
train_idx = [i for i in range(len(x_train_poisoned))]
import random
random.seed(0)
train_idx_shuffle = random.shuffle(train_idx)
x_train_poisoned = x_train_poisoned[train_idx_shuffle][0]
y_train_poisoned = y_train_poisoned[train_idx_shuffle][0]

In [20]:
# determine which of the test data falls into the cluster
x_test = [i[0] for i in test_loader]
y_test = [i[1] for i in test_loader]

In [21]:
# modified version of randeigen
# instead of giving a robust aggregate over the sample, we return the samples which are removed
# take in a set of inputs and their corresponding gradient
# output the clean inputs 
import math

def power_iteration(mat, iterations, device):
    dim = mat.shape[0]
    u = torch.randn((dim, 1)).to(device)
    for _ in range(iterations):
        u = mat @ u / torch.linalg.norm(mat @ u) 
    eigenvalue = u.T @ mat @ u
    return eigenvalue, u

# return the index of the clean samples
def randomized_agg_forced(data,raw_data,label, eps_poison=0.2, eps_jl=0.1, eps_pow = 0.1, seed=12):
    n = int(data.shape[0])
    feature_shape = data[0].shape
    n_dim = int(np.prod(np.array(feature_shape)))
    res =  _randomized_agg(data,raw_data,label, eps_poison, eps_jl, eps_pow, 1, 10**-5, forced=True, seed=seed) # set threshold for convergence as 1*10**-5 (i.e. float point error)
    return res

def _randomized_agg(data, raw_data, label, eps_poison=0.2, eps_jl=0.1, eps_pow = 0.1, threshold = 20, clean_eigen = 10**-5, device = device, forced=False, seed=None):
    if seed:
        torch.manual_seed(seed)
    
    n = int(data.shape[0])
    data = data.to(device)
    
    d = int(math.prod(data[0].shape))
    data_flatten = data.reshape(n, d)
    data_mean = torch.mean(data_flatten, dim=0)
    data_sd = torch.std(data_flatten, dim=0)
    data_norm = (data_flatten - data_mean)/data_sd
    
    k = min(int(math.log(d)//eps_jl**2), d)
    
    A = torch.randn((d, k)).to(device)
    A = A/(k**0.5)

    Y = data_flatten @ A # n times k
    Y = Y.to(device)
    power_iter_rounds = int(- math.log(4*k)/(2*math.log(1-eps_pow)))
    clean_eigen = clean_eigen * d/k
    old_eigenvalue = None
    for _ in range(max(int(eps_poison*n), 10)):
        Y_mean = torch.mean(Y, dim=0)
        Y = (Y - Y_mean)
        Y_cov = torch.cov(Y.T)
        Y_sq = Y_cov
            
        eigenvalue, eigenvector = power_iteration(Y_sq, power_iter_rounds, device)

        proj_Y = torch.abs(Y @ eigenvector )
        proj_Y = torch.flatten(proj_Y)
        # if old_eigenvalue:
            # print("old_eigenvalue: " + str(old_eigenvalue))
            # print("new_eigenvalue: " + str(eigenvalue))
            # print("ratio: " + str(abs(old_eigenvalue - eigenvalue)/old_eigenvalue))
        if forced and old_eigenvalue and abs(old_eigenvalue - eigenvalue)/old_eigenvalue < 10**-4:
            # print('converge')
            break
        else: 
            old_eigenvalue = eigenvalue
        
        if len(data) <= (1-4*eps_poison) * n:
            # print('new_criteria')
            break 
        
        
        uniform_rand = torch.rand(proj_Y.shape).to(device)
        kept_idx = uniform_rand > (proj_Y/torch.max(proj_Y))
        Y = Y[kept_idx]
        data = data[kept_idx]
        raw_data = raw_data[kept_idx]
        label = label[kept_idx]
    return raw_data, label
    

In [22]:
x_poisoned_flat = [i.flatten().cpu().numpy() for i in x_train_poisoned]
poisoned_cluster_labels = kmeans.predict(x_poisoned_flat)

In [23]:
x_test_cluster = kmeans.predict([i.detach().cpu().numpy().flatten() for i in x_test])
x_test_in_poison = [int(i in n_poison_each_cluster) for i in x_test_cluster]

In [24]:
sum(x_test_in_poison)

326

In [25]:
import statistics 
import copy
torch.manual_seed(1) # for reproducibility... 
n_epoch = 30
batch_size = 50
for _ in range(n_epoch):
    print("epoch " + str(_))
    x_train_poisoned_copy = copy.deepcopy(x_train_poisoned)
    y_train_poisoned_copy = copy.deepcopy(y_train_poisoned)
    curr_grads = []
    for i in range(0,len(x_train_poisoned),batch_size):

        net_copy = copy.deepcopy(network)
        optimizer_copy = copy.deepcopy(optimizer)
        feature = x_train_poisoned_copy[i:i+batch_size]
        feature.requires_grad = True  ### CRUCIAL LINE !!!
        target = y_train_poisoned_copy[i:i+batch_size]
        optimizer_copy.zero_grad()
        output = net_copy(feature)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer_copy.step()
        curr_grads.extend(feature.grad)

        
#     print(curr_grads[0])
#     # purify training set
    grad_flatten = torch.stack(curr_grads).flatten(start_dim=1)
    x_train_purified, y_train_purified = randomized_agg_forced(grad_flatten, x_train_poisoned_copy, y_train_poisoned_copy)
    print(len(x_train_purified))
    # train on purifired set
    for i in range(0,len(x_train_purified),batch_size):

        feature = x_train_purified[i:i+batch_size]
        target = y_train_purified[i:i+batch_size]
        optimizer.zero_grad()
        output = network(feature)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    
    
    
    with torch.no_grad():
        ncorrect_clean = 0
        ncorrect_poison = 0
        
        n_clean = 0
        n_poison = 0
        out_prob = network(torch.stack([i[0] for i in x_test]).to(device))
        out_class = torch.argmax(out_prob, dim=-1)
        for i in range(len(out_class)):
            pred_class = int(out_class[i])
            actual_class = int(y_test[i])
        
            # belongs to our poison subpopulation
            if x_test_in_poison[i]:
                ncorrect_poison += int(pred_class == actual_class)
                n_poison += 1
            else:
                ncorrect_clean += int(pred_class == actual_class)
                n_clean += 1
        clean_acc = ncorrect_clean/n_clean
        asr = 1-ncorrect_poison/n_poison # propotion we managed to misclassify as adversary class (either by chance or attack)
        print("ACC: ", clean_acc, "ASR", asr)

epoch 0
5870
ACC:  0.323340913789539 ASR 0.9141104294478528
epoch 1
5223
ACC:  0.46909241265247054 ASR 0.9631901840490797
epoch 2
5279
ACC:  0.9122389911101922 ASR 0.6042944785276074
epoch 3
5763
ACC:  0.5653297498449452 ASR 0.7852760736196319
epoch 4
5753
ACC:  0.8960099235063056 ASR 0.6349693251533742
epoch 5
5610
ACC:  0.8306801736613604 ASR 0.7730061349693251
epoch 6
5631
ACC:  0.8618978705809386 ASR 0.6411042944785277
epoch 7
5037
ACC:  0.8723382261732479 ASR 0.665644171779141
epoch 8
6298
ACC:  0.9360140583005996 ASR 0.5337423312883436
epoch 9
6224
ACC:  0.9715732892288609 ASR 0.16564417177914115
epoch 10
5570
ACC:  0.9767417820963407 ASR 0.1288343558282209
epoch 11
5335
ACC:  0.9773620012404383 ASR 0.11963190184049077
epoch 12
6203
ACC:  0.9790159189580319 ASR 0.10122699386503065
epoch 13
5208
ACC:  0.9783956998139343 ASR 0.07055214723926384
epoch 14
5057
ACC:  0.9784990696712839 ASR 0.042944785276073594
epoch 15
6223
ACC:  0.9788091792433327 ASR 0.039877300613496924
epoch 16
60